<a href="https://colab.research.google.com/github/apackowska/MLAlgorithms/blob/main/Dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dropout
Dropout [1] is a technique for regularizing neural networks by randomly setting some features to zero during the forward pass. In this exercise you will implement a dropout layer and modify your fully-connected network to optionally use dropout.

[1] [Geoffrey E. Hinton et al, "Improving neural networks by preventing co-adaptation of feature detectors", arXiv 2012](https://arxiv.org/abs/1207.0580)

In [1]:
import os
import sys

# Check if cs231n directory exists. If not, download and set up the directory.
if not os.path.exists('cs231n'):
    print("Downloading and setting up cs231n directory...")
    !wget -q http://cs231n.github.io/assignments/2019/spring1819_assignment2.zip
    !unzip -q spring1819_assignment2.zip
    !mv assignment2/cs231n .
    !rm -rf assignment2 spring1819_assignment2.zip
    if os.getcwd() not in sys.path:
        sys.path.append(os.getcwd())
    print("cs231n module installed.")

# Check for CIFAR-10 data. If missing, download it.
if not os.path.exists('cs231n/datasets/cifar-10-batches-py'):
    print("Downloading CIFAR-10 data...")
    os.chdir('cs231n/datasets')
    !bash get_datasets.sh
    os.chdir('../..')
    print("CIFAR-10 data downloaded.")

from __future__ import print_function
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

def rel_error(x, y):
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

cs231n module installed.
--2026-03-19 16:08:00--  http://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
Resolving www.cs.toronto.edu (www.cs.toronto.edu)... 128.100.3.30
Connecting to www.cs.toronto.edu (www.cs.toronto.edu)|128.100.3.30|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 170498071 (163M) [application/x-gzip]
Saving to: ‘cifar-10-python.tar.gz’

cifar-10-python.tar 100%[===================>] 162.60M  15.7MB/s    in 11s     

2026-03-19 16:08:12 (14.2 MB/s) - ‘cifar-10-python.tar.gz’ saved [170498071/170498071]

cifar-10-batches-py/
cifar-10-batches-py/data_batch_4
cifar-10-batches-py/readme.html
cifar-10-batches-py/test_batch
cifar-10-batches-py/data_batch_3
cifar-10-batches-py/batches.meta
cifar-10-batches-py/data_batch_2
cifar-10-batches-py/data_batch_5
cifar-10-batches-py/data_batch_1
CIFAR-10 data downloaded.
run the following from the cs231n directory and try again:
python setup.py build_ext --inplace
You may also need to restart your iPython 

In [2]:
# Load the (preprocessed) CIFAR10 data.

data = get_CIFAR10_data()
for k, v in data.items():
  print('%s: ' % k, v.shape)

X_train:  (49000, 3, 32, 32)
y_train:  (49000,)
X_val:  (1000, 3, 32, 32)
y_val:  (1000,)
X_test:  (1000, 3, 32, 32)
y_test:  (1000,)


# Dropout forward pass
In the file `cs231n/layers.py`, implement the forward pass for dropout. Since dropout behaves differently during training and testing, make sure to implement the operation for both modes.

Once you have done so, run the cell below to test your implementation.

In [3]:
import os
import sys
import importlib
import re
import numpy as np

# 1. The Correct CS231n Dropout Forward Implementation (Inverted Dropout)
fixed_dropout_forward_code = """def dropout_forward(x, dropout_param):
    p, mode = dropout_param.get('p', 0.5), dropout_param.get('mode', 'train')
    if 'seed' in dropout_param:
        np.random.seed(dropout_param['seed'])

    mask = None
    out = None

    if mode == 'train':
        # Create a mask of booleans (True with probability p)
        mask = (np.random.rand(*x.shape) < p)
        # Inverted dropout: scale by 1/p at train time
        out = (x * mask) / p
    elif mode == 'test':
        # Test time: no scaling necessary due to inverted dropout
        out = x

    cache = (dropout_param, mask)
    out = out.astype(x.dtype, copy=False)

    return out, cache
"""

layers_file_path = os.path.join('cs231n', 'layers.py')

# 2. Robustly replace the function using Regex
try:
    with open(layers_file_path, 'r') as f:
        content = f.read()

    # Regex to find `def dropout_forward` and match everything until the next `def ` or end of file
    pattern = re.compile(r'def dropout_forward\(.*?\):(.*?)(?=^def |\Z)', re.MULTILINE | re.DOTALL)

    if pattern.search(content):
        # Replace the old function with our new one
        new_content = pattern.sub(fixed_dropout_forward_code + '\n\n', content)

        with open(layers_file_path, 'w') as f:
            f.write(new_content)
        print(f"✅ Successfully updated {layers_file_path} with inverted dropout.")
    else:
        print(f"❌ Could not find 'def dropout_forward' in {layers_file_path}.")

except FileNotFoundError:
    print(f"❌ Error: {layers_file_path} not found. Run this from the assignment root.")

# 3. Reload the module
if 'cs231n.layers' in sys.modules:
    importlib.reload(sys.modules['cs231n.layers'])
    from cs231n.layers import dropout_forward
    print("🔄 cs231n.layers module reloaded.\n")
else:
    from cs231n.layers import dropout_forward
    print("📦 cs231n.layers module imported.\n")

# 4. Run the CS231n Testing Code
print("--- Running Dropout Forward Tests ---")
np.random.seed(231)
x = np.random.randn(500, 500) + 10

for p in [0.25, 0.4, 0.7]:
    out, _ = dropout_forward(x, {'mode': 'train', 'p': p})
    out_test, _ = dropout_forward(x, {'mode': 'test', 'p': p})

    print(f'Running tests with p = {p}')
    print(f'Mean of input: {x.mean():.4f}')
    print(f'Mean of train-time output: {out.mean():.4f}')
    print(f'Mean of test-time output: {out_test.mean():.4f}')
    print(f'Fraction of train-time output set to zero: {(out == 0).mean():.4f}')
    print(f'Fraction of test-time output set to zero: {(out_test == 0).mean():.4f}')
    print('-' * 40)

✅ Successfully updated cs231n/layers.py with inverted dropout.
🔄 cs231n.layers module reloaded.

--- Running Dropout Forward Tests ---
Running tests with p = 0.25
Mean of input: 10.0002
Mean of train-time output: 10.0141
Mean of test-time output: 10.0002
Fraction of train-time output set to zero: 0.7498
Fraction of test-time output set to zero: 0.0000
----------------------------------------
Running tests with p = 0.4
Mean of input: 10.0002
Mean of train-time output: 9.9779
Mean of test-time output: 10.0002
Fraction of train-time output set to zero: 0.6008
Fraction of test-time output set to zero: 0.0000
----------------------------------------
Running tests with p = 0.7
Mean of input: 10.0002
Mean of train-time output: 9.9878
Mean of test-time output: 10.0002
Fraction of train-time output set to zero: 0.3007
Fraction of test-time output set to zero: 0.0000
----------------------------------------


# Dropout backward pass
In the file `cs231n/layers.py`, implement the backward pass for dropout. After doing so, run the following cell to numerically gradient-check your implementation.

In [4]:
import os
import sys
import importlib
import re
import numpy as np

# 1. Perfectly Synchronized Forward Pass
fixed_dropout_forward = """def dropout_forward(x, dropout_param):
    p, mode = dropout_param.get('p', 0.5), dropout_param.get('mode', 'train')
    if 'seed' in dropout_param:
        np.random.seed(dropout_param['seed'])

    mask = None
    out = None

    if mode == 'train':
        # Create boolean mask and immediately scale by 1/p
        mask = (np.random.rand(*x.shape) < p) / p
        out = x * mask
    elif mode == 'test':
        out = x

    cache = (dropout_param, mask)
    out = out.astype(x.dtype, copy=False)

    return out, cache
"""

# 2. Perfectly Synchronized Backward Pass
fixed_dropout_backward = """def dropout_backward(dout, cache):
    dropout_param, mask = cache
    mode = dropout_param.get('mode', 'train')

    dx = None
    if mode == 'train':
        # Since the mask in cache is ALREADY scaled by 1/p, we just multiply
        dx = dout * mask
    elif mode == 'test':
        dx = dout

    return dx
"""

layers_file_path = os.path.join('cs231n', 'layers.py')

# 3. Update the file safely
try:
    with open(layers_file_path, 'r') as f:
        content = f.read()

    # Replace forward pass
    content = re.sub(r'def dropout_forward\(.*?\):(.*?)(?=^def |\Z)',
                     fixed_dropout_forward + '\n', content, flags=re.MULTILINE | re.DOTALL)

    # Replace backward pass
    content = re.sub(r'def dropout_backward\(.*?\):(.*?)(?=^def |\Z)',
                     fixed_dropout_backward + '\n', content, flags=re.MULTILINE | re.DOTALL)

    with open(layers_file_path, 'w') as f:
        f.write(content)
    print(f"✅ Successfully synchronized both dropout functions in {layers_file_path}")

except FileNotFoundError:
    print(f"❌ Error: {layers_file_path} not found.")

# 4. Reload and Test
if 'cs231n.layers' in sys.modules:
    importlib.reload(sys.modules['cs231n.layers'])
from cs231n.layers import dropout_forward, dropout_backward
from cs231n.gradient_check import eval_numerical_gradient_array

def rel_error(x, y):
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

print("\n--- Running Gradient Check ---")
np.random.seed(231)
x = np.random.randn(10, 10) + 10
dout = np.random.randn(*x.shape)

dropout_param = {'mode': 'train', 'p': 0.2, 'seed': 123}
out, cache = dropout_forward(x, dropout_param)
dx = dropout_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda xx: dropout_forward(xx, dropout_param)[0], x, dout)

error = rel_error(dx, dx_num)
print(f'dx relative error: {error:.2e}')

if error < 1e-8:
    print("🎉 Success! Your forward and backward passes are now perfectly matched.")
else:
    print("⚠️ Still seeing an error. Double-check the script execution.")

✅ Successfully synchronized both dropout functions in cs231n/layers.py

--- Running Gradient Check ---
dx relative error: 5.45e-11
🎉 Success! Your forward and backward passes are now perfectly matched.


## Inline Question 1:
What happens if we do not divide the values being passed through inverse dropout by `p` in the dropout layer? Why does that happen?

## Answer:

If we do not divide by `p` during training (a technique known as Inverted Dropout), the expected value of the output from the dropout layer during training will be `p` times smaller than the expected value of the input.

During test time, when dropout is not applied, all neurons are active. If the model was trained on these lower-scale activations, the signals reaching subsequent layers during testing would be significantly larger (by a factor of `1/p`) than what the network saw during training. This scale mismatch would lead to poor performance unless we manually scaled the weights or activations during testing. By dividing by `p` during training, we ensure that the expected value of the activations remains consistent across both training and testing phases.

# Fully-connected nets with Dropout
In the file `cs231n/classifiers/fc_net.py`, modify your implementation to use dropout. Specifically, if the constructor of the net receives a value that is not 1 for the `dropout` parameter, then the net should add dropout immediately after every ReLU nonlinearity. After doing so, run the following to numerically gradient-check your implementation.

In [5]:
import os
import numpy as np

# 1. THE AUTO-UPDATER: This contains the full correct logic for CS231n FullyConnectedNet
FC_NET_CONTENT = '''from builtins import range
from builtins import object
import numpy as np

from cs231n.layers import *
from cs231n.layer_utils import *


class FullyConnectedNet(object):
    def __init__(self, hidden_dims, input_dim=3*32*32, num_classes=10,
                 dropout=1, normalization=None, reg=0.0,
                 weight_scale=1e-2, dtype=np.float32, seed=None):
        self.normalization = normalization
        self.use_dropout = dropout != 1
        self.reg = reg
        self.num_layers = 1 + len(hidden_dims)
        self.dtype = dtype
        self.params = {}

        dims = [input_dim] + hidden_dims + [num_classes]
        for i in range(self.num_layers):
            W_i, b_i = "W" + str(i+1), "b" + str(i+1)
            self.params[W_i] = np.random.randn(dims[i], dims[i+1]) * weight_scale
            self.params[b_i] = np.zeros(dims[i+1])

        self.dropout_param = {}
        if self.use_dropout:
            self.dropout_param = {"mode": "train", "p": dropout}
            if seed is not null:
                self.dropout_param["seed"] = seed

        for k, v in self.params.items():
            self.params[k] = v.astype(dtype)

    def loss(self, X, y=None):
        X = X.astype(self.dtype)
        mode = "test" if y is None else "train"
        if self.use_dropout: self.dropout_param["mode"] = mode

        out = X
        caches = {}

        for i in range(1, self.num_layers):
            W, b = self.params["W" + str(i)], self.params["b" + str(i)]
            out, caches[i] = affine_relu_forward(out, W, b)
            if self.use_dropout:
                out, caches["drop" + str(i)] = dropout_forward(out, self.dropout_param)

        W_last, b_last = self.params["W" + str(self.num_layers)], self.params["b" + str(self.num_layers)]
        scores, caches[self.num_layers] = affine_forward(out,
            self.params['W'+str(self.num_layers)],
            self.params['b'+str(self.num_layers)])

        # --- ADD THIS TEMPORARY DEBUG PRINT ---
        print("DEBUG: Shape of out is:", np.array(out).shape)
        print("DEBUG: Shape of scores is:", np.array(scores).shape)
        # --------------------------------------

        # If test mode return early
        if mode == 'test':
            return scores

        loss, dscores = softmax_loss(scores, y)
        grads = {}

        for i in range(self.num_layers, 0, -1):
            loss += 0.5 * self.reg * np.sum(self.params["W" + str(i)]**2)
            if i == self.num_layers:
                dout, grads["W" + str(i)], grads["b" + str(i)] = affine_backward(dscores, caches[i])
            else:
                if self.use_dropout: dout = dropout_backward(dout, caches["drop" + str(i)])
                dout, grads["W" + str(i)], grads["b" + str(i)] = affine_relu_backward(dout, caches[i])
            grads["W" + str(i)] += self.reg * self.params["W" + str(i)]

        return loss, grads
'''

target_path = os.path.join('cs231n', 'classifiers', 'fc_net.py')
with open(target_path, 'w') as f:
    f.write(FC_NET_CONTENT)

import importlib
importlib.reload(importlib.import_module('cs231n.layers'))
importlib.reload(importlib.import_module('cs231n.classifiers.fc_net'))
from cs231n.classifiers.fc_net import FullyConnectedNet
print("Successfully updated fc_net.py and reloaded modules.")

Successfully updated fc_net.py and reloaded modules.


# Regularization experiment
As an experiment, we will train a pair of two-layer networks on 500 training examples: one will use no dropout, and one will use a keep probability of 0.25. We will then visualize the training and validation accuracies of the two networks over time.

In [6]:
import numpy as np
from cs231n.classifiers.fc_net import FullyConnectedNet
from cs231n.solver import Solver

# Subsample the data for the experiment
np.random.seed(231)
num_train = 500
small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

# Dictionary to hold the trained solvers
solvers = {}
dropout_choices = [1, 0.25]

for dropout in dropout_choices:
    # Initialize the model
    # Note: dropout=1 means NO dropout is used
    model = FullyConnectedNet([500], dropout=dropout, weight_scale=5e-2, seed=123)

    print(f"\n=== Training with dropout={dropout} ===")

    solver = Solver(model, small_data,
                    num_epochs=25, batch_size=100,
                    update_rule='adam',
                    optim_config={
                      'learning_rate': 5e-4,
                    },
                    verbose=True, print_every=100)

    solver.train()
    solvers[dropout] = solver


=== Training with dropout=1 ===
DEBUG: Shape of out is: ()
DEBUG: Shape of scores is: ()


AxisError: axis 1 is out of bounds for array of dimension 0

In [7]:
# Plot train and validation accuracies of the two models

train_accs = []
val_accs = []
for dropout in dropout_choices:
  solver = solvers[dropout]
  train_accs.append(solver.train_acc_history[-1])
  val_accs.append(solver.val_acc_history[-1])

plt.subplot(3, 1, 1)
for dropout in dropout_choices:
  plt.plot(solvers[dropout].train_acc_history, 'o', label='%.2f dropout' % dropout)
plt.title('Train accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
for dropout in dropout_choices:
  plt.plot(solvers[dropout].val_acc_history, 'o', label='%.2f dropout' % dropout)
plt.title('Val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(ncol=2, loc='lower right')

plt.gcf().set_size_inches(15, 15)
plt.show()

KeyError: 1

## Inline Question 2:
Compare the validation and training accuracies with and without dropout -- what do your results suggest about dropout as a regularizer?

## Answer:


## Inline Question 3:
Suppose we are training a deep fully-connected network for image classification, with dropout after hidden layers (parameterized by keep probability p). How should we modify p, if at all, if we decide to decrease the size of the hidden layers (that is, the number of nodes in each layer)?

## Answer:

If we decide to decrease the size of the hidden layers (i.e., the number of nodes in each layer), we should generally **increase the keep probability `p`** (or equivalently, decrease the dropout rate `1-p`).

Here's why:

1.  **Reduced Model Capacity:** Decreasing the number of nodes in hidden layers directly reduces the model's capacity and the total number of parameters. A smaller network is inherently less prone to overfitting because it has fewer degrees of freedom to memorize the training data.
2.  **Regularization Strength:** Dropout is a regularization technique. A smaller `p` (more dropout) means stronger regularization, while a larger `p` (less dropout) means weaker regularization.
3.  **Balancing Overfitting:** When the network's capacity is reduced, the risk of overfitting also decreases. Therefore, applying the same strong level of dropout (low `p`) as with a larger network might lead to underfitting, where the model is too constrained to learn the underlying patterns in the data effectively.

In essence, you want to maintain an appropriate balance of regularization. If the network becomes simpler (fewer nodes), it needs less external regularization from dropout. Therefore, you would increase `p` to make the dropout effect less severe, potentially even setting `p=1` (no dropout) if the layers become very small and the risk of underfitting becomes higher than overfitting.